# Step 3 — Learned affinity + tracksdata graph solver

Goal: candidate edges (7 µm physical radius) -> a small learned affinity model scoring
those candidates -> `tracksdata`'s graph solver (`max_children=2` for divisions) -> the
actual competition metric (`score = adjusted_edge_jaccard + 0.1 * division_jaccard`),
computed locally via the re-vendored official `tracking_cellmot` package.

**Scope**: `44b6` only, one sample (`44b6_0db75fae`, same as notebooks 01/02), one
10-frame window. This is a smoke-test-scale prototype of the whole pipeline
end-to-end, not a production model.

**Why a separate contiguous-frame segmentation run (Section 0)**: Step 1's checkpointed
masks are 3 widely-spaced frames (`t=0,29,58`), chosen to match GT-labeled validation
frames. That's fine for Step 2's token-extraction functions but useless for learning
frame-to-frame affinity, which needs real adjacent-frame masks. So this notebook segments
its own short contiguous window first.

## 0. Setup + contiguous-window segmentation

Reuses notebook 01's crop+Cellpose-SAM approach (`do_3D=False`, `stitch_threshold=0.25`,
GT-centered crop windows) but for `T_START..T_END` **consecutive** frames instead of
widely-spaced ones. `crop_window_moving` centers each frame's crop on its own GT points
when available, or carries the previous frame's offset forward otherwise (sparse labeling
means not every frame in a contiguous window is guaranteed to have GT) -- for this
particular sample/window every frame happens to have GT coverage, so it behaves like
per-frame GT-centering throughout.

Checkpointed the same way as notebook 01 (`.npz` per frame) so this ~70-90 min CPU run
only needs to happen once.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import zarr
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.measure import regionprops_table, marching_cubes, mesh_surface_area
from scipy.ndimage import center_of_mass

import tracksdata as td
from tracksdata.constants import DEFAULT_ATTR_KEYS as K
from tracksdata.edges import DistanceEdges, GenericFuncEdgeAttrs
from tracksdata.solvers import NearestNeighborsSolver

from tracking_cellmot.io import save_graph
from tracking_cellmot.metrics import evaluate, per_sample_metrics, summarise, node_recall

assert td.__version__ == "0.1.0rc9", td.__version__  # tracking_cellmot's pyproject pins a different git ref; guard against it silently winning

SCALE_Z, SCALE_Y, SCALE_X = 1.625, 0.40625, 0.40625
SCALE = np.array([SCALE_Z, SCALE_Y, SCALE_X])
VOXEL_VOLUME_UM3 = SCALE_Z * SCALE_Y * SCALE_X
MAX_MATCH_DIST_UM = 7.0

DATA_ROOT = "/mnt/d/KAGGLE/data/extracted/train"
SAMPLE_ID = "44b6_0db75fae"
T_START, T_END = 29, 38  # inclusive, 10 consecutive frames, brackets notebook 01's t=29 checkpoint

TEST_IDS = {"44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"}
assert SAMPLE_ID not in TEST_IDS

CHECKPOINT_DIR = Path("_checkpoints/03_tracking_solver")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def load_geff(sid):
    g = zarr.open_group(f"{DATA_ROOT}/{sid}.geff", mode="r")
    nodes = pd.DataFrame(
        {
            "node_id": g["nodes/ids"][:],
            "t": g["nodes/props/t/values"][:],
            "z": g["nodes/props/z/values"][:],
            "y": g["nodes/props/y/values"][:],
            "x": g["nodes/props/x/values"][:],
        }
    )
    edges = pd.DataFrame(g["edges/ids"][:], columns=["source", "target"])
    return nodes, edges


def match_centroids(pred_um, gt_um, max_dist=MAX_MATCH_DIST_UM):
    if len(pred_um) == 0 or len(gt_um) == 0:
        return [], list(range(len(pred_um))), list(range(len(gt_um)))
    cost = cdist(pred_um, gt_um)
    row_ind, col_ind = linear_sum_assignment(cost)
    matches = [(r, c) for r, c in zip(row_ind, col_ind) if cost[r, c] <= max_dist]
    matched_pred = {r for r, _ in matches}
    matched_gt = {c for _, c in matches}
    fp = [r for r in range(len(pred_um)) if r not in matched_pred]
    fn = [c for c in range(len(gt_um)) if c not in matched_gt]
    return matches, fp, fn


nodes_df, edges_df = load_geff(SAMPLE_ID)
img_group = zarr.open_group(f"{DATA_ROOT}/{SAMPLE_ID}.zarr", mode="r")
arr = img_group["0"]
print("volume shape (T,Z,Y,X):", arr.shape, "| GT nodes:", len(nodes_df), "| GT edges:", len(edges_df))

In [ ]:
from cellpose import models

STITCH_THRESHOLD = 0.25
MAX_Z, MAX_Y, MAX_X = 24, 144, 144
MARGIN_Z, MARGIN_YX = 6, 24


def crop_window(center, margin, cap, extent):
    half = min(margin, cap // 2)
    lo = int(np.clip(center - half, 0, max(0, extent - 2 * half)))
    hi = min(extent, lo + 2 * half)
    return lo, hi


def crop_window_moving(nodes_df, t, prev_offset, margin_z, margin_yx, cap_z, cap_yx, extent):
    """Center the crop on frame t's own GT points if available; otherwise carry the
    previous frame's offset forward (sparse labeling means not every frame in a
    contiguous window is guaranteed to have GT)."""
    Z, Y, X = extent
    gt_rows = nodes_df[nodes_df["t"] == t]
    if len(gt_rows) > 0:
        z_center = (gt_rows["z"].min() + gt_rows["z"].max()) / 2
        y_center = (gt_rows["y"].min() + gt_rows["y"].max()) / 2
        x_center = (gt_rows["x"].min() + gt_rows["x"].max()) / 2
        z0, z1 = crop_window(z_center, margin_z, cap_z, Z)
        y0, y1 = crop_window(y_center, margin_yx, cap_yx, Y)
        x0, x1 = crop_window(x_center, margin_yx, cap_yx, X)
        return (z0, y0, x0), (z1, y1, x1)
    if prev_offset is not None:
        z0, y0, x0 = prev_offset
        half_z, half_yx = min(margin_z, cap_z // 2), min(margin_yx, cap_yx // 2)
        return (z0, y0, x0), (z0 + 2 * half_z, y0 + 2 * half_yx, x0 + 2 * half_yx)
    raise ValueError(f"no GT at t={t} and no previous offset to carry forward")


model = models.CellposeModel(gpu=False)

masks_by_t = {}
offsets_by_t = {}
prev_offset = None
Z, Y, X = arr.shape[1:]

for t in range(T_START, T_END + 1):
    ckpt_path = CHECKPOINT_DIR / f"44b6_contig_t{t}.npz"
    if ckpt_path.exists():
        data = np.load(ckpt_path)
        masks, offset = data["masks"], tuple(int(v) for v in data["offset"])
        print(f"t={t}: loaded cached masks -> {masks.max()} instances")
    else:
        offset, end = crop_window_moving(
            nodes_df, t, prev_offset, MARGIN_Z, MARGIN_YX, MAX_Z, MAX_Y, (Z, Y, X)
        )
        z0, y0, x0 = offset
        z1, y1, x1 = end
        vol = np.asarray(arr[t, z0:z1, y0:y1, x0:x1])
        t0 = time.time()
        masks, flows, styles = model.eval(
            vol, do_3D=False, stitch_threshold=STITCH_THRESHOLD, z_axis=0, diameter=None
        )
        dt = time.time() - t0
        np.savez(ckpt_path, masks=masks, offset=np.array(offset))
        print(f"t={t}: crop {vol.shape} offset={offset} -> {masks.max()} instances in {dt:.1f}s")
    masks_by_t[t] = masks
    offsets_by_t[t] = offset
    prev_offset = offset

print(f"\nsegmented {len(masks_by_t)} contiguous frames t={T_START}..{T_END}")

## 1. GT-stand-in sanity check (before touching real masks)

Build a `tracksdata` graph directly from GT points (skip Cellpose entirely), run
`DistanceEdges` + `NearestNeighborsSolver(max_children=2)` on it (the old classical
baseline's exact pattern, `git show HEAD:claude_test/src/classical/track.py`), and check
how many real GT edges a pure nearest-neighbor solver recovers when given perfect
detections. This validates the tracksdata plumbing and bounds what distance-only linking
can achieve -- it is **not** the real metric (predicted nodes here ARE the GT nodes).

In [ ]:
def build_gt_stand_in_graph(nodes_df):
    graph = td.graph.RustWorkXGraph()
    for key in (K.Z, K.Y, K.X, "z_phys", "y_phys", "x_phys"):
        graph.add_node_attr_key(key, pl.Float64)

    node_id_map = {}  # gt node_id -> tracksdata node id
    for row in nodes_df.itertuples():
        td_id = graph.add_node(
            {
                K.T: int(row.t),
                K.Z: float(row.z), K.Y: float(row.y), K.X: float(row.x),
                "z_phys": float(row.z) * SCALE_Z,
                "y_phys": float(row.y) * SCALE_Y,
                "x_phys": float(row.x) * SCALE_X,
            }
        )
        node_id_map[row.node_id] = td_id
    return graph, node_id_map


gt_stand_in_graph, node_id_map = build_gt_stand_in_graph(nodes_df)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    DistanceEdges(
        distance_threshold=7.0, n_neighbors=5, attr_keys=["z_phys", "y_phys", "x_phys"]
    ).add_edges(gt_stand_in_graph)
    solved_stand_in = NearestNeighborsSolver(max_children=2, edge_weight=K.EDGE_DIST).solve(gt_stand_in_graph)

print("graph nodes:", gt_stand_in_graph.num_nodes(), "| candidate edges:", gt_stand_in_graph.num_edges())
print("solved nodes:", solved_stand_in.num_nodes(), "| solved edges:", solved_stand_in.num_edges())

solved_edge_attrs = solved_stand_in.edge_attrs(attr_keys=[])
solved_pairs = set(zip(solved_edge_attrs[K.EDGE_SOURCE].to_list(), solved_edge_attrs[K.EDGE_TARGET].to_list()))

gt_pairs = {
    (node_id_map[row.source], node_id_map[row.target])
    for row in edges_df.itertuples()
    if row.source in node_id_map and row.target in node_id_map
}
recovered = solved_pairs & gt_pairs
recovered_fraction = len(recovered) / len(gt_pairs)
print(f"GT edges: {len(gt_pairs)} | solved edges: {len(solved_pairs)} | recovered: {len(recovered)} ({recovered_fraction:.1%})")
assert 0 < recovered_fraction <= 1.0, "sanity check failed: expected a non-trivial recovered-edge fraction"

## 2. Real-mask tokens (Step 2 functions, copy-adapted)

Same token-extraction functions as `02_cell_tokens.ipynb`, applied to this notebook's
contiguous-window masks. Instances flagged `is_flat_degenerate` (single-Z-plane stitch
failures, undefined shape features) are dropped here -- unlike notebook 02 which just
flags them for reporting, Step 3 needs every node to have usable numeric features for the
affinity MLP, and a degenerate mask is likely a spurious detection anyway.

In [ ]:
def compute_morphology_features(mask, spacing=SCALE):
    props = regionprops_table(
        mask, spacing=tuple(spacing),
        properties=("label", "area", "axis_major_length", "axis_minor_length", "solidity", "bbox"),
    )
    df = pd.DataFrame(props).rename(columns={"area": "volume_um3"})
    bbox_cols = [c for c in df.columns if c.startswith("bbox-")]
    extents = np.stack([df[f"bbox-{3 + i}"] - df[f"bbox-{i}"] for i in range(3)], axis=1)
    df["is_flat_degenerate"] = (extents == 1).any(axis=1)
    df = df.drop(columns=bbox_cols)

    surface_areas = []
    for label in df["label"]:
        submask = mask == label
        padded = np.pad(submask, 1)
        try:
            verts, faces, _, _ = marching_cubes(padded.astype(np.float32), level=0.5, spacing=tuple(spacing))
            sa = mesh_surface_area(verts, faces)
        except (RuntimeError, ValueError):
            sa = np.nan
        surface_areas.append(sa)
    df["surface_area_um2"] = surface_areas
    df["elongation"] = df["axis_major_length"] / df["axis_minor_length"].replace(0, np.nan)
    df["sphericity"] = (np.pi ** (1 / 3)) * (6 * df["volume_um3"]) ** (2 / 3) / df["surface_area_um2"]

    shape_cols = ["elongation", "sphericity", "solidity"]
    df.loc[df["is_flat_degenerate"], shape_cols] = np.nan

    return df[["label", "volume_um3", "surface_area_um2", *shape_cols, "is_flat_degenerate"]]


def compute_intensity_features(mask, intensity_vol):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    rows = []
    for label in labels:
        vals = intensity_vol[mask == label].astype(np.float64)
        p10, p50, p90 = np.percentile(vals, [10, 50, 90])
        rows.append(
            {
                "label": int(label),
                "mean_intensity": vals.mean(),
                "std_intensity": vals.std(),
                "p10": p10, "p50": p50, "p90": p90,
                "integrated_intensity": vals.sum(),
            }
        )
    return pd.DataFrame(rows)


def compute_voxel_centroids(mask, offset=(0, 0, 0)):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    if len(labels) == 0:
        return pd.DataFrame(columns=["label", "z_vox", "y_vox", "x_vox"])
    centers = center_of_mass(mask, labels=mask, index=labels)
    centers = np.asarray(centers) + np.asarray(offset)
    df = pd.DataFrame(centers, columns=["z_vox", "y_vox", "x_vox"])
    df.insert(0, "label", labels)
    df["z_um"] = df["z_vox"] * SCALE_Z
    df["y_um"] = df["y_vox"] * SCALE_Y
    df["x_um"] = df["x_vox"] * SCALE_X
    return df


def compute_image_placeholder_features(mask, intensity_vol, n_bins=16, hist_range=None):
    labels = np.unique(mask)
    labels = labels[labels != 0]
    if hist_range is None:
        hist_range = (float(intensity_vol.min()), float(intensity_vol.max()))
    rows = []
    for label in labels:
        vals = intensity_vol[mask == label].astype(np.float64)
        hist, _ = np.histogram(vals, bins=n_bins, range=hist_range, density=True)
        row = {"label": int(label)}
        row.update({f"hist_{i}": h for i, h in enumerate(hist)})
        rows.append(row)
    return pd.DataFrame(rows)


def build_token_table(masks_by_t, offsets_by_t, intensity_by_t, sample_id):
    frames = []
    for t, mask in masks_by_t.items():
        offset = offsets_by_t[t]
        intensity_vol = intensity_by_t[t]
        morph = compute_morphology_features(mask)
        inten = compute_intensity_features(mask, intensity_vol)
        pos = compute_voxel_centroids(mask, offset)
        img_feat = compute_image_placeholder_features(mask, intensity_vol)
        token = morph.merge(inten, on="label").merge(pos, on="label").merge(img_feat, on="label")
        token.insert(0, "t", t)
        frames.append(token)
    table = pd.concat(frames, ignore_index=True)
    table.insert(0, "sample_id", sample_id)
    return table


intensity_by_t = {}
for t, mask in masks_by_t.items():
    z0, y0, x0 = offsets_by_t[t]
    zs, ys, xs = mask.shape
    intensity_by_t[t] = np.asarray(arr[t, z0 : z0 + zs, y0 : y0 + ys, x0 : x0 + xs])

token_table_contig = build_token_table(masks_by_t, offsets_by_t, intensity_by_t, SAMPLE_ID)
n_degenerate = int(token_table_contig["is_flat_degenerate"].sum())
print(f"{len(token_table_contig)} instances across {len(masks_by_t)} frames, {n_degenerate} flagged is_flat_degenerate (dropped below)")

token_table_contig = token_table_contig[~token_table_contig["is_flat_degenerate"]].reset_index(drop=True)
token_table_contig.to_parquet(CHECKPOINT_DIR / f"{SAMPLE_ID}_contig_tokens.parquet", index=False)
print(f"kept {len(token_table_contig)} non-degenerate instances")
token_table_contig.head()

## 3. Candidate edges (7 µm physical radius)

One `tracksdata` node per token row (all feature columns as node attrs), then
`DistanceEdges` on the physical-µm centroid columns.

In [ ]:
FEATURE_COLS = [
    c for c in token_table_contig.columns
    if c not in ("sample_id", "t", "label", "is_flat_degenerate")
]
print(f"{len(FEATURE_COLS)} feature columns:", FEATURE_COLS)


def build_candidate_graph_from_tokens(token_df, feature_cols):
    graph = td.graph.RustWorkXGraph()
    for key in (K.Z, K.Y, K.X, *feature_cols):
        graph.add_node_attr_key(key, pl.Float64)
    nodes = []
    for row in token_df.itertuples(index=False):
        row_d = row._asdict()
        attrs = {
            K.T: int(row_d["t"]),
            K.Z: float(row_d["z_vox"]), K.Y: float(row_d["y_vox"]), K.X: float(row_d["x_vox"]),
        }
        for c in feature_cols:
            attrs[c] = float(row_d[c])
        nodes.append(attrs)
    node_ids = graph.bulk_add_nodes(nodes)
    return graph, node_ids


candidate_graph, contig_node_ids = build_candidate_graph_from_tokens(token_table_contig, FEATURE_COLS)
print("graph nodes:", candidate_graph.num_nodes())

DistanceEdges(
    distance_threshold=7.0, n_neighbors=5, attr_keys=["z_um", "y_um", "x_um"]
).add_edges(candidate_graph)
print("candidate edges:", candidate_graph.num_edges())

## 4. GT-masked edge labels

Because GT is temporally sparse in general, a candidate edge can only be labeled when
*both* endpoints' frames have GT coverage: match each endpoint independently to the
nearest GT node at its own frame (reusing the same scaled bipartite matching as Step 1's
validation), then:
- **positive**: both endpoints matched to GT nodes AND that GT pair is a true GT edge.
- **negative**: both matched, but not a true GT edge.
- **excluded** (`label=None`): a frame has no GT coverage at all, or an endpoint fails to
  match any GT node (ambiguous -- could be a real unannotated cell, not a confirmed
  negative).

In [ ]:
def label_candidate_edges(graph, gt_nodes_df, gt_edges_df, max_dist=MAX_MATCH_DIST_UM):
    node_attrs = graph.node_attrs(attr_keys=[K.NODE_ID, K.T, K.Z, K.Y, K.X])
    edge_attrs = graph.edge_attrs(attr_keys=[])
    gt_edge_pairs = set(zip(gt_edges_df["source"], gt_edges_df["target"]))

    pred_to_gt = {}
    for t in node_attrs[K.T].unique().to_list():
        frame_pred = node_attrs.filter(pl.col(K.T) == t)
        gt_rows = gt_nodes_df[gt_nodes_df["t"] == t]
        if len(gt_rows) == 0:
            continue
        pred_um = frame_pred.select(K.Z, K.Y, K.X).to_numpy() * SCALE
        gt_um = gt_rows[["z", "y", "x"]].to_numpy(dtype=float) * SCALE
        matches, _, _ = match_centroids(pred_um, gt_um, max_dist=max_dist)
        pred_ids = frame_pred[K.NODE_ID].to_list()
        gt_ids = gt_rows["node_id"].to_list()
        for r, c in matches:
            pred_to_gt[pred_ids[r]] = gt_ids[c]

    labels = []
    for row in edge_attrs.iter_rows(named=True):
        src, tgt = row[K.EDGE_SOURCE], row[K.EDGE_TARGET]
        gt_src, gt_tgt = pred_to_gt.get(src), pred_to_gt.get(tgt)
        if gt_src is None or gt_tgt is None:
            labels.append(None)
        else:
            labels.append(1 if (gt_src, gt_tgt) in gt_edge_pairs else 0)

    return edge_attrs.with_columns(pl.Series("label", labels, dtype=pl.Int8))


labeled_edges = label_candidate_edges(candidate_graph, nodes_df, edges_df)
class_counts = labeled_edges["label"].value_counts().sort("label")
print(class_counts)
n_pos = int((labeled_edges["label"] == 1).sum())
n_neg = int((labeled_edges["label"] == 0).sum())
n_excluded = int(labeled_edges["label"].is_null().sum())
print(f"positive: {n_pos} | negative: {n_neg} | excluded: {n_excluded} (of {len(labeled_edges)} candidate edges)")

**Observed on this window**: 11 positive / 0 negative / 11 excluded. The labeled set has
**zero negatives** -- this window's candidate edges (28 nodes, 22 candidates within 7 µm)
happen to mostly resolve to either a true GT match or an unmatched (excluded) node, with no
case of "two GT-matched cells that aren't actually linked." This is a real limitation of
prototyping on one 10-frame window with only ~2-3 tracked cells, not a bug in the labeling
logic (verified separately against a synthetic case in Section 4's design). It means the
affinity MLP below has nothing to learn to discriminate against and will simply converge to
"always positive" -- expected and fine for validating the training -> scoring -> solving
plumbing, but not evidence the model has learned anything. Revisit once training spans more
samples/frames.

## 5. Affinity MLP

A small MLP scores `(Z_{i,t}, Z_{j,t+1})` pairs: input is `[src_features, tgt_features,
delta_t, distance_um]`, output is a raw logit trained with `BCEWithLogitsLoss` on the
Section 4 labels (excluded edges are simply not in the training set). Frame-pair-level
train/val split (the last 2 source frames held out), not a random row split, so the same
instance can't leak across train/val. This is a smoke-test-scale run (one sample, one
window) to validate the training -> scoring -> solving pipeline, not a real model.

In [ ]:
class AffinityMLP(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)  # raw logit


def edge_feature_vector(src_attrs, tgt_attrs, feature_cols):
    delta_t = tgt_attrs[K.T] - src_attrs[K.T]
    dz = tgt_attrs["z_um"] - src_attrs["z_um"]
    dy = tgt_attrs["y_um"] - src_attrs["y_um"]
    dx = tgt_attrs["x_um"] - src_attrs["x_um"]
    distance_um = (dz ** 2 + dy ** 2 + dx ** 2) ** 0.5
    vec = [src_attrs[c] for c in feature_cols] + [tgt_attrs[c] for c in feature_cols] + [delta_t, distance_um]
    return np.array(vec, dtype=np.float32)


def build_training_data(graph, labeled_edges, feature_cols):
    node_attrs = graph.node_attrs(attr_keys=[K.NODE_ID, K.T, *feature_cols]).to_pandas().set_index(K.NODE_ID)
    rows = labeled_edges.filter(pl.col("label").is_not_null())
    X, y, src_t = [], [], []
    for r in rows.iter_rows(named=True):
        src_attrs = node_attrs.loc[r[K.EDGE_SOURCE]].to_dict()
        tgt_attrs = node_attrs.loc[r[K.EDGE_TARGET]].to_dict()
        X.append(edge_feature_vector(src_attrs, tgt_attrs, feature_cols))
        y.append(r["label"])
        src_t.append(src_attrs[K.T])
    return np.stack(X), np.array(y, dtype=np.float32), np.array(src_t)


X, y, src_t = build_training_data(candidate_graph, labeled_edges, FEATURE_COLS)
print("training pairs:", X.shape, "| positives:", int(y.sum()), "| negatives:", int((y == 0).sum()))

unique_ts = sorted(set(src_t.tolist()))
val_ts = set(unique_ts[-2:])
train_mask = np.array([t not in val_ts for t in src_t])
print(f"frame-pair split: train source-frames {sorted(set(src_t[train_mask].tolist()))}, val source-frames {sorted(val_ts)}")

# normalize features using train-split statistics only
feat_mean = X[train_mask].mean(axis=0)
feat_std = X[train_mask].std(axis=0) + 1e-6
X_norm = (X - feat_mean) / feat_std

X_train, y_train = torch.from_numpy(X_norm[train_mask]), torch.from_numpy(y[train_mask])
X_val, y_val = torch.from_numpy(X_norm[~train_mask]), torch.from_numpy(y[~train_mask])

torch.manual_seed(0)
affinity_model = AffinityMLP(in_dim=X.shape[1])
optimizer = torch.optim.Adam(affinity_model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(50):
    affinity_model.train()
    optimizer.zero_grad()
    loss = loss_fn(affinity_model(X_train), y_train)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0 or epoch == 49:
        affinity_model.eval()
        with torch.no_grad():
            val_loss = loss_fn(affinity_model(X_val), y_val).item() if len(y_val) else float("nan")
        print(f"epoch {epoch:2d} | train loss {loss.item():.4f} | val loss {val_loss:.4f}")

In [ ]:
affinity_model.eval()


def mlp_score_func(src_attrs, tgt_attrs):
    vec = edge_feature_vector(src_attrs, tgt_attrs, FEATURE_COLS)
    vec_norm = (vec - feat_mean) / feat_std
    with torch.no_grad():
        logit = affinity_model(torch.from_numpy(vec_norm).unsqueeze(0))[0]
        prob = torch.sigmoid(logit).item()
    return 1.0 - prob  # invert: NearestNeighborsSolver treats LOWER edge_weight as better


GenericFuncEdgeAttrs(
    func=mlp_score_func, attr_keys=[K.T, *FEATURE_COLS], output_key="ml_weight"
).add_edge_attrs(candidate_graph)

ml_weights = candidate_graph.edge_attrs(attr_keys=["ml_weight"])["ml_weight"]
print(f"scored {len(ml_weights)} candidate edges | ml_weight range: [{ml_weights.min():.3f}, {ml_weights.max():.3f}]")

## 6. Solve

`NearestNeighborsSolver(max_children=2, edge_weight="ml_weight")` -- the solver sorts
candidate edges ascending by weight and greedily accepts them (confirmed in
`tracksdata/solvers/_nearest_neighbors_solver.py`: `np.argsort(weights)`), hence the
`1 - prob` inversion above.

In [ ]:
solved_pred = NearestNeighborsSolver(max_children=2, edge_weight="ml_weight").solve(candidate_graph)
print("solved nodes:", solved_pred.num_nodes(), "| solved edges:", solved_pred.num_edges())

solved_pred_nodes = solved_pred.node_attrs()
solved_pred_edges = solved_pred.edge_attrs()
solved_pred_nodes.head(), solved_pred_edges.head()

## 7. Score with the official metric

Load the real `.geff` as a `tracksdata` graph, restrict it to this window's `t` range,
and call the re-vendored `tracking_cellmot.metrics.evaluate` -> `per_sample_metrics` ->
`summarise` directly -- the first point in the whole pipeline where the actual
competition-formula score is computed locally.

In [ ]:
solved_pred_detached = solved_pred.detach()  # evaluate()'s division scoring needs .copy(), unsupported on GraphView

_gt_full = td.graph.IndexedRXGraph.from_geff(f"{DATA_ROOT}/{SAMPLE_ID}.geff")
gt_full_graph = _gt_full[0] if isinstance(_gt_full, tuple) else _gt_full
gt_window_graph = gt_full_graph.filter(
    td.attrs.NodeAttr(K.T) >= T_START, td.attrs.NodeAttr(K.T) <= T_END
).subgraph()
print("GT window nodes/edges:", gt_window_graph.num_nodes(), gt_window_graph.num_edges())

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    er = evaluate(solved_pred_detached, gt_window_graph, scale=tuple(SCALE), max_distance=7.0)
    nr = node_recall(solved_pred_detached, gt_window_graph)

print("EvaluationResult:", er)
row = per_sample_metrics(er, n_total=float(gt_window_graph.num_nodes()), node_recall=nr)
print("per_sample_metrics:", row)
summary = summarise([row])
print("\n=== Local competition-metric score (Step 3 MVP prototype) ===")
for k, v in summary.items():
    print(f"  {k}: {v}")

In [ ]:
pred_geff_path = CHECKPOINT_DIR / "pred_geffs" / f"{SAMPLE_ID}_contig.geff"
save_graph(solved_pred_detached, pred_geff_path, overwrite=True)

torch.save(affinity_model.state_dict(), CHECKPOINT_DIR / "affinity_mlp.pt")
import json
with open(CHECKPOINT_DIR / "affinity_mlp_config.json", "w") as f:
    json.dump(
        {
            "feature_cols": FEATURE_COLS,
            "in_dim": int(X.shape[1]),
            "feat_mean": feat_mean.tolist(),
            "feat_std": feat_std.tolist(),
        },
        f,
    )
print(f"saved solved prediction -> {pred_geff_path}")
print(f"saved model + config -> {CHECKPOINT_DIR / 'affinity_mlp.pt'}, {CHECKPOINT_DIR / 'affinity_mlp_config.json'}")

## 8. Next steps

Explicitly deferred (not built in this MVP):
- **Division-probability head**: this window's GT has no divisions to train/validate
  against (confirmed separately: this sample has 0 GT divisions overall), so `max_children=2`
  on the greedy solver remains the entire division model for now.
- **`ILPSolver`**: a global-optimum alternative to the greedy `NearestNeighborsSolver`,
  worth trying once there's a real model to solve with, not just this smoke test.
- **Real `e_image`**: Cellpose's `styles` output is dead (Section notes in notebook 02);
  a real SAM-ViT-encoder embedding via `model.net.encoder` is a scoped future upgrade.
- **`6bba`**: segmentation there currently fails locally (Step 1: 0 precision/recall) --
  an explicit out-of-scope follow-up, not addressed here.
- **Scale beyond one sample/window**: this whole notebook ran on 1 sample x 10 frames.
  Scaling up means running Section 0's segmentation over more of `dataset_splits.json`'s
  (filtered to what's actually on disk, and excluding the 4 official `TEST_IDS`) train
  pool -- expensive locally (CPU-only), realistically a Kaggle-GPU-notebook task.